# Theme: From Understanding Humans → To Guiding Them

In [1]:
# Import Libraries
import pandas as pd
import numpy as np
from scipy.sparse import hstack

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report


## Train

In [2]:
# Load the Train Data
df = pd.read_csv('/content/Sample_arvyax_reflective_dataset.xlsx - Dataset_120.csv')

In [3]:
# Perform EDA, Remove Null Data as there are small portion of it, or cuz of Text
# Reset the Index
df = df.dropna().drop('id', axis=1).reset_index(drop=True)

In [4]:
nltk.download('wordnet')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [5]:
wordnet =WordNetLemmatizer()
stopword = set(stopwords.words('english'))

In [6]:
import re

In [7]:
# Function for Cleaning Text Data
def preprocess_text(text):
  text = re.sub(r'[^a-zA-Z\s]',' ',text)
  text = text.lower()
  tokens = word_tokenize(text)
  tokens = [wordnet.lemmatize(word)for word in tokens if not word in stopword]
  return ' '.join(tokens)

In [8]:
# Store Cleaned Text in seprate Column
df['clean_text'] = df['journal_text'].apply(preprocess_text)

In [9]:
# WOrd Embedding -- Word ->Vector
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_text = vectorizer.fit_transform(df['clean_text'])

In [10]:
# Encode Categorical Column
df = pd.get_dummies(df, columns=[
    'ambience_type',
    'previous_day_mood',
    'face_emotion_hint',
    'reflection_quality',
    'time_of_day'
])

In [11]:
# Standardisation
scaler =StandardScaler()
num_cols = ['duration_min','sleep_hours','energy_level','stress_level']
df[num_cols] = scaler.fit_transform(df[num_cols])

In [12]:
# Final Matrix for training

X_others = df.drop(columns=['journal_text','clean_text','emotional_state','intensity'])
X = hstack((X_text, X_others.astype(float))) # Indenpendent
# Dependent
y_state = df['emotional_state']
y_intensity = df['intensity']

In [13]:
# Dimentionality Reduction
svd = TruncatedSVD(n_components=100,n_iter=5, random_state=42)
X_reduced = svd.fit_transform(X)

In [14]:
# Train - Train Testt Split
X_train, X_val, y_train, y_val = train_test_split(X_reduced, y_state, test_size=0.2, random_state=42)
_, _, y_train_intensity, y_val_intensity = train_test_split(X_reduced, y_intensity, test_size=0.2, random_state=42)

In [15]:
# Emotional State Model
model_state = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)
model_state.fit(X_train, y_train)

RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=42)

In [16]:
# Evaluate the Model
y_pred_state = model_state.predict(X_val)

accuracy_score = accuracy_score(y_val, y_pred_state)
print(accuracy_score)

0.5518867924528302


In [17]:
# INTENSITY MODEL
model_intensity = RandomForestRegressor(n_estimators=100, random_state=42)
model_intensity.fit(X_train, y_train_intensity)


# EVALUATE REGRESSION

y_pred_intensity = model_intensity.predict(X_val)

mse = mean_squared_error(y_val_intensity, y_pred_intensity)
print("MSE:", mse)

MSE: 1.9543169811320755


# Test

In [18]:
df_test = pd.read_csv('/content/arvyax_test_inputs_120.xlsx - Sheet1.csv')
df_test.head(2)

,id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality
0,10001,woke up feeling more organized mentally. i was...,cafe,4,8.5,3,1,night,mixed,happy_face,vague
1,10002,started off distracted most of the time. this ...,mountain,4,8.5,1,2,afternoon,mixed,happy_face,clear


In [19]:
df_test = df_test.drop('id', axis=1).reset_index(drop=True)

In [20]:
df_test['clean_text'] = df_test['journal_text'].apply(preprocess_text)
X_test_text = vectorizer.transform(df_test['clean_text'])

In [21]:
df_test = pd.get_dummies(df_test, columns=[
    'ambience_type',
    'previous_day_mood',
    'face_emotion_hint',
    'reflection_quality',
    'time_of_day'
])
num_cols = ['duration_min','sleep_hours','energy_level','stress_level']
df_test[num_cols] = scaler.transform(df_test[num_cols])
X_others_test = df_test.drop(columns=['journal_text','clean_text'])

In [22]:
# align the column
X_others_test = X_others_test.reindex(columns=X_others.columns, fill_value=0)

In [23]:
X_test = hstack((X_test_text, X_others_test.astype(float))) # Actual Test Data

In [24]:
X_test_reduced= svd.transform(X_test)

In [25]:
# Actual PREDICTIONS on given Dataset

y_test_state_pred = model_state.predict(X_test_reduced)
y_test_intensity_pred = model_intensity.predict(X_test_reduced)

In [26]:
#  CONFIDENCE SCORE

probs = model_state.predict_proba(X_test_reduced)
confidence = np.max(probs, axis=1)

In [27]:
# Uncertinity flag
uncertain_flag = (confidence < 0.6).astype(int)

In [28]:
# Decision Layer
def decision_layer(state, intensity):
    intensity = round(intensity)


    action = "reflect"
    timing = "later_today"

    if state in ["stress", "anxiety"]:
        if intensity >= 4:
            action = "breathing exercise"
            timing = "now"
        elif intensity >= 2:
            action = "short walk"
            timing = "within_15_min"
        else:
            action = "journaling"
            timing = "later_today"

    elif state in ["sad", "sadness"]:
        if intensity >= 4:
            action = "talk to someone"
            timing = "tonight"
        else:
            action = "light journaling"
            timing = "later_today"

    elif state in ["happy", "calm"]:
        action = "continue routine"
        timing = "now"

    elif state in ["tired", "low_energy"]:
        action = "rest"
        timing = "now"

    elif state == "neutral":
        if intensity >= 3:
            action = "deep work"
            timing = "within_15_min"
        else:
            action = "plan tasks"
            timing = "later_today"

    return action, timing

In [29]:
# Support Message
def generate_message(state, intensity, action, timing):
    intensity = round(intensity)

    return f"You seem {state} with intensity {intensity}. " \
           f"It might help to try {action} {timing.replace('_', ' ')}."

In [30]:
results = []

for i in range(X_test_reduced.shape[0]):

    state = y_test_state_pred[i]
    intensity = float(y_test_intensity_pred[i])
    conf = float(confidence[i])
    uncertain = int(uncertain_flag[i])

    # Decision layer
    action, timing = decision_layer(state, intensity)

    # Support message
    message = generate_message(state, intensity, action, timing)

    results.append({
        "predicted_state": state,
        "predicted_intensity": round(intensity, 2),
        "action": action,
        "timing": timing,
        "confidence": round(conf, 3),
        "uncertain_flag": uncertain,
        "message": message
    })

final_df = pd.DataFrame(results)
print(final_df.head())

  predicted_state  predicted_intensity     action         timing  confidence  \
0         focused                 3.51    reflect    later_today       0.260   
1         neutral                 2.51  deep work  within_15_min       0.242   
2         focused                 3.36    reflect    later_today       0.350   
3         neutral                 3.17  deep work  within_15_min       0.228   
4         neutral                 3.06  deep work  within_15_min       0.312   

   uncertain_flag                                            message  
0               1  You seem focused with intensity 4. It might he...  
1               1  You seem neutral with intensity 3. It might he...  
2               1  You seem focused with intensity 3. It might he...  
3               1  You seem neutral with intensity 3. It might he...  
4               1  You seem neutral with intensity 3. It might he...  


In [ ]:
# FINAL CSV EXPORT
final_df.to_csv("predictions.csv", index=False)

In [31]:
# Serialize the MOdel
import os
import joblib

# Save Model
joblib.dump(model_state,'model_state.joblib')
joblib.dump(model_intensity,'model_intensity.joblib')

# Save Preprocessing
joblib.dump(vectorizer,'tfidf_vectorizer.joblib')
joblib.dump(scaler,'scaler.joblib')
joblib.dump(svd,'svd.joblib')

# Save Featue columns oreder
joblib.dump(X_others.columns.tolist(),'feature_columns.joblib')


['feature_columns.joblib']

In [32]:
# app.py - a Minimal Flask Rest API
app_code = '''
import flask
from flask import Flask,  request, jsonify
import joblib

app = Flask(__name__)
# Load The MODELs
model_state = joblib.load('model_state.joblib')
model_intensity = joblib.load('model_intensity.joblib')
vectorizer = joblib.load('tfidf_vectorizer.joblib')
scaler = joblib.load('scaler.joblib')
feature_columns = joblib.load('feature_columns.joblib')


# NLP SetUp
wordnet = WordNetLemmatizer()
stopword = set(stopwords.words('english'))

def preprocess_text(text):
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [wordnet.lemmatize(word) for word in tokens if word not in stopword]
    return ' '.join(tokens)

 # Decision Layer
def decision_layer(state, intensity):
   intensity = round(float(intensity))

   if state in ['stress', 'anxiety']:
     return "breathing exercise", "now"
   elif state in ["sad"]:
     return "talk to someone", "tonight"
   elif state in ["happy"]:
     return "continue routine", "now"
   else:
     return "reflect", "later_today"


 # API Route
@app.route("/")
def home():
    return "Emotion-Based Decision API is running 🚀"

@app.route('/predict',methods = ['POST'])
def predict():
   data = request.get_json(force=True)
   # Convert data into DataFrame
   df = pd.DataFrame([data])

   # Preprocess text
   df['clean_text'] = df['journal_text'].apply(preprocess_text)
   #TF-IDF
   X_text = vectorizer.transform(df['clean_text'])

   # REMOVE TEXT COLUMNS
   df = df.drop(columns=["journal_text", "clean_text"], errors='ignore')

   # One-hot Encode
   df = pd.get_dummies(df)
   # Align Columns
   df = df.reindex(columns = feature_columns, fill_value=0)

   #Scale numeric
   num_cols = ['duration_min','sleep_hours','energy_level','stress_level']
   df[num_cols] = scaler.transform(df[num_cols])

   # Combine
   X = hstack((X_text, df.astype(float)))
   # Truncated SVD
   X = svd.transform(X)

   # Predictions
   state = model_state.predict(X)[0]
   intensity = float(model_intensity.predict(X)[0])

   # Confidence
   probs = model_state.predict_proba(X)
   confidence = float(np.max(probs))

   uncertain_flag = int(confidence<0.6)

   # Decision
   action, timing = decision_layer(state, intensity)

   return jsonify({
         "predicted_state": state,
         "predicted_intensity": round(intensity, 2),
         "confidence": round(confidence, 3),
         "uncertain_flag": uncertain_flag,
         "what_to_do": action,
         "when_to_do": timing
     })

if __name__ == "__main__":
    app.run(host = 0.0.0.0, port = 5000)

'''
with open('app.py', 'w') as f:
    f.write(app_code)

print('Wrote app.py')

Wrote app.py


<>:21: SyntaxWarning: invalid escape sequence '\s'
<>:21: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_7372/1443341896.py:21: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub(r'[^a-zA-Z\s]', ' ', text)


In [33]:
# Requirement.txt
!pip freeze > requirement.txt

In [34]:
with open('requirement.txt','w')as f:
  f.write('''
  flask
  pandas
  numpy
  scikit-learn
  re
  nltk
  scipy
  gunicorn
  ''')

In [35]:
from google.colab import files
files.download('requirement.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
with open("Procfile", "w") as f:
    f.write("web: gunicorn app:app")

In [37]:
from google.colab import files
files.download('Procfile')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>